# QC: `ss_subclass_v5_nounlabeled_nn` -> `ss_subclass_nounlabeled_nmm_v5_nn`

Independent re-derivation of everything `Nonmam_Mapping_SAMap_final.ipynb` wrote, checked against
the four h5ads. Nothing here writes to disk and no h5ad is loaded into memory (`backed='r'` only).

Each section prints `PASS` / `FAIL`; the last cell collects them into one table.

| # | check |
|---|---|
| A | consensus partition recomputed from the 12 pairwise CSVs == `nonmam_consensus_assignment` |
| B | `nonmam_consensus_groups` members / cells / min-score agree with the assignment CSV |
| C | per-cell: `OUT_LEVEL == LEVEL.map(result)` in all four h5ads |
| D | node coverage: h5ad species-cluster labels <-> pairwise-CSV rows |
| E | safety: no NN / Allen-named / foreign-prefix label was pulled into a group |
| F | edge robustness: reciprocal-score symmetry, best hits sitting near `THRESH` |
| G | composition sanity: within-species dupes, per-species cell balance, group sizes |
| H | (optional) `nonmam_samap_blocks` cache re-derives the same best hits |

In [ ]:
import os, pickle, itertools
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import anndata as ad

# ---- must match Nonmam_Mapping_SAMap_final.ipynb section 1 -------------------
LEVEL     = 'ss_subclass_v5_nounlabeled_nn'
OUT_LEVEL = 'ss_subclass_nounlabeled_nmm_v5_nn'
THRESH    = 0.2
DATE      = '09072026'

ROOT      = '/home/prateek/docker/'
SAM_DIR   = ROOT + 'Active_SAM_joined/'
SAMAP_DIR = ROOT + 'Active_SAMap_Joined/active_samap/Non-mammal/'
BLOCKS    = SAMAP_DIR + f'nonmam_samap_blocks_{DATE}.pkl'

H5AD = {
    'cj': SAM_DIR + 'SAM_CJ_joined_v2_cleaned_03122025.h5ad',
    'ac': SAM_DIR + 'SAM_AC_ncbi_soupx_cleaned_03122025.h5ad',
    'xt': SAM_DIR + 'SAM_XT_joined_Slc17a6_cleaned_03122205_nostale.h5ad',
    'dr': SAM_DIR + 'SAM_DR_ncbi_joined_cleaned_07172026.h5ad',
}
SPECIES = ['cj', 'ac', 'xt', 'dr']
PAIRS   = [('cj','ac'), ('cj','xt'), ('cj','dr'), ('ac','xt'), ('ac','dr'), ('xt','dr')]

RESULTS = {}                       # check id -> (ok, note)
def report(cid, ok, note=''):
    RESULTS[cid] = (bool(ok), note)
    print(('PASS  ' if ok else 'FAIL  ') + cid + ('   ' + note if note else ''))
    return ok

for f in list(H5AD.values()) + [BLOCKS]:
    print(('OK   ' if os.path.exists(f) else 'MISS '), os.path.basename(f))

## Load the written artefacts

In [ ]:
mapping_frames = {}
for a, b in PAIRS:
    for src, dst in ((a, b), (b, a)):
        fn = SAMAP_DIR + f'{src}_{dst}_nonmam_mapping_SAMap_{DATE}.csv'
        mapping_frames[(src, dst)] = pd.read_csv(fn, index_col=0)

assign_csv = pd.read_csv(SAMAP_DIR + f'nonmam_consensus_assignment_SAMap_{DATE}.csv',
                         index_col=0)['group']
groups_csv = pd.read_csv(SAMAP_DIR + f'nonmam_consensus_groups_SAMap_{DATE}.csv')

# node -> species, taken from the CSV row labels (the same universe the run used)
node_species = {}
for (src, _), df in mapping_frames.items():
    node_species.update({i: src for i in df.index})

print(f'{len(mapping_frames)} pairwise tables, {len(node_species)} nodes '
      f'({ {s: sum(v == s for v in node_species.values()) for s in SPECIES} })')
print(f'assignment CSV: {len(assign_csv)} types in {assign_csv.nunique()} groups; '
      f'groups CSV: {len(groups_csv)} rows')

## A. Recompute the consensus from the pairwise CSVs

Same rule as the source notebook: an edge is a **reciprocal best hit** with `score > THRESH`,
groups are connected components of size > 1.

Group *names* are compared separately from group *membership*. `name_groups` numbers components
with `sorted(comps, key=len, reverse=True)`, which does not break ties between equal-sized
components deterministically (the components come out of `nx.connected_components` as sets), so
the counter suffix can permute between runs. Membership is what matters; a name permutation is
reported as a note, not a failure.

In [ ]:
def build_graph(mapping_frames, thresh=THRESH):
    G = nx.Graph()
    G.add_nodes_from(node_species)
    for a, b in PAIRS:
        for src, dst in ((a, b), (b, a)):
            fwd, rev = mapping_frames[(src, dst)], mapping_frames[(dst, src)]
            for u, row in fwd.iterrows():
                v, score = row['SAMAP match'], float(row['SAMAP score'])
                if v in rev.index and rev.loc[v, 'SAMAP match'] == u and score > thresh:
                    G.add_edge(u, v, weight=score)
    return G


def name_groups(g):
    comps = [c for c in nx.connected_components(g) if len(c) > 1]
    counters, res, members = defaultdict(int), {}, {}
    for c in sorted(comps, key=len, reverse=True):
        combo = '_'.join(sorted({node_species[n] for n in c}, key=SPECIES.index))
        counters[combo] += 1
        name = f'{combo}_{counters[combo]}'
        members[name] = sorted(c)
        for n in c:
            res[n] = name
    return res, members


G = build_graph(mapping_frames)
result, members = name_groups(G)
print(f'edges {G.number_of_edges()}, groups {len(members)}, assigned types {len(result)}')

part_new = {frozenset(m) for m in members.values()}
part_ref = {frozenset(g.index) for _, g in assign_csv.groupby(assign_csv)}

same_members = part_new == part_ref
name_diffs   = {n: (result.get(n), assign_csv.get(n))
                for n in set(result) | set(assign_csv.index)
                if result.get(n) != assign_csv.get(n)}

note = 'identical names' if not name_diffs else (
    f'{len(name_diffs)} types differ by group NAME only (counter tie) '
    f'-- membership identical' if same_members else f'{len(name_diffs)} name diffs')
report('A. partition reproducible from pairwise CSVs', same_members, note)

if not same_members:
    print('\n  only in recomputed:', [sorted(c) for c in part_new - part_ref])
    print('  only in CSV       :', [sorted(c) for c in part_ref - part_new])
elif name_diffs:
    print('\n  name permutations (harmless, but the h5ad labels are tied to the CSV naming):')
    for n, (a_, b_) in sorted(name_diffs.items()):
        print(f'    {n:<8} recomputed {a_:<14} CSV {b_}')

## B. `nonmam_consensus_groups` self-consistency

In [ ]:
part_grp = {frozenset(m.split()) for m in groups_csv['members']}
ok_b1 = part_grp == part_ref
print(('OK  ' if ok_b1 else 'DIFF'), 'groups CSV members == assignment CSV partition')

# recompute cells / species / dupes / min-score straight from the h5ads + graph
cell_counts = {}
for s in SPECIES:
    a = ad.read_h5ad(H5AD[s], backed='r')
    vc = a.obs[LEVEL].astype(str).value_counts()
    cell_counts.update({n: int(vc.get(n, 0)) for n in node_species if node_species[n] == s})
    del a

rows = []
for _, r in groups_csv.iterrows():
    mem = r['members'].split()
    sub = G.subgraph(mem)
    rows.append({
        'group': r['group'],
        'types_csv': r['types'],      'types_calc': len(mem),
        'cells_csv': r['cells'],      'cells_calc': sum(cell_counts[n] for n in mem),
        'species_csv': r['species'],  'species_calc': len({node_species[n] for n in mem}),
        'min_csv': r['min score'],
        'min_calc': round(min(d['weight'] for *_, d in sub.edges(data=True)), 3),
        'connected': nx.is_connected(sub),
    })
chk_b = pd.DataFrame(rows)
bad_b = chk_b[(chk_b.types_csv != chk_b.types_calc) | (chk_b.cells_csv != chk_b.cells_calc)
              | (chk_b.species_csv != chk_b.species_calc)
              | ~np.isclose(chk_b.min_csv, chk_b.min_calc, atol=1e-3) | ~chk_b.connected]

report('B. groups CSV consistent (members/cells/species/min-score/connected)',
       ok_b1 and bad_b.empty, f'{len(bad_b)} bad rows')
display(bad_b if not bad_b.empty else chk_b.head())

## C. Per-cell check against the four h5ads

The only thing `apply_labels` does is `src.map(lambda x: result.get(x, x))`. Re-apply the
*written* assignment CSV to `LEVEL` and demand an exact match with `OUT_LEVEL`, cell by cell.

In [ ]:
res_map = assign_csv.to_dict()
per_cell = []
mismatch_examples = {}

for s in SPECIES:
    a = ad.read_h5ad(H5AD[s], backed='r')
    missing_cols = [c for c in (LEVEL, OUT_LEVEL) if c not in a.obs.columns]
    if missing_cols:
        per_cell.append({'species': s, 'cells': a.n_obs, 'mismatched': -1,
                         'note': 'missing ' + ', '.join(missing_cols)})
        del a
        continue

    src = a.obs[LEVEL].astype(str)
    out = a.obs[OUT_LEVEL].astype(str)
    exp = src.map(lambda x: res_map.get(x, x))
    bad = exp != out

    per_cell.append({
        'species'    : s,
        'cells'      : len(src),
        'mismatched' : int(bad.sum()),
        'relabelled' : int((out != src).sum()),
        'pct_relab'  : round(100 * (out != src).mean(), 2),
        'n_in'       : src.nunique(),
        'n_out'      : out.nunique(),
        'out_is_cat' : isinstance(a.obs[OUT_LEVEL].dtype, pd.CategoricalDtype),
        'nan_out'    : int(a.obs[OUT_LEVEL].isna().sum()),
        'note'       : '',
    })
    if bad.any():
        mismatch_examples[s] = (pd.DataFrame({'in': src[bad], 'written': out[bad],
                                              'expected': exp[bad]})
                                .drop_duplicates().head(20))
    del a

per_cell = pd.DataFrame(per_cell)
report('C. per-cell relabelling matches the assignment CSV',
       (per_cell['mismatched'] == 0).all(),
       f"{int(per_cell['mismatched'].clip(lower=0).sum())} mismatched cells")
display(per_cell)
for s, ex in mismatch_examples.items():
    print(f'\n--- {s} distinct mismatches ---'); display(ex)

## D. Node coverage

Every `<sp>_<n>` label present in the h5ad must appear as a row in that species' pairwise tables
(otherwise it silently never got a chance to form an edge), and vice versa.

In [ ]:
def is_species_cluster(item, prefix):
    return len(item) > 3 and item[:2] == prefix and item[2] == '_' and item[3] != 'm'


cov = []
for s in SPECIES:
    a = ad.read_h5ad(H5AD[s], backed='r')
    labs = set(a.obs[LEVEL].astype(str).unique())
    del a
    h5_nodes  = {i for i in labs if is_species_cluster(i, s)}
    csv_nodes = {n for n, sp in node_species.items() if sp == s}
    cov.append({'species': s,
                'h5ad nodes': len(h5_nodes), 'csv nodes': len(csv_nodes),
                'in h5ad not CSV': sorted(h5_nodes - csv_nodes),
                'in CSV not h5ad': sorted(csv_nodes - h5_nodes),
                'grouped': len(csv_nodes & set(res_map)),
                'left alone': sorted(csv_nodes - set(res_map))})
cov = pd.DataFrame(cov)
report('D. node coverage h5ad <-> pairwise CSVs',
       all(not r['in h5ad not CSV'] and not r['in CSV not h5ad'] for _, r in cov.iterrows()))
pd.set_option('display.max_colwidth', 250)
display(cov[['species', 'h5ad nodes', 'csv nodes', 'grouped', 'in h5ad not CSV', 'in CSV not h5ad']])
print('\nnodes left with their original label (no reciprocal partner above THRESH):')
for _, r in cov.iterrows():
    print(f"  {r['species']}: {r['left alone']}")

## E. Safety of the node filter

`is_species_cluster` is the only filter actually applied — the source notebook defines
`is_neuronal` (an `endswith('NN')` test) but never calls it. That is harmless *only if* no
species-prefixed label carries an NN suffix, and if merged clusters (`dr_5_14`) are meant to be
eligible. This cell checks that explicitly, plus that no Allen-named subclass, `Unlabeled`, or a
foreign species' prefix ever entered a group.

In [ ]:
issues = []
for s in SPECIES:
    a = ad.read_h5ad(H5AD[s], backed='r')
    labs = sorted(set(a.obs[LEVEL].astype(str).unique()))
    del a
    for i in labs:
        node = is_species_cluster(i, s)
        if node and i.endswith('NN'):
            issues.append((s, i, 'NN-suffixed label passes is_species_cluster -> treated as neuronal node'))
        if node and i.count('_') > 1:
            issues.append((s, i, 'merged/multi-underscore cluster used as a node (check intent)'))
        if not node and i in res_map:
            issues.append((s, i, 'non-node label carries a group assignment'))
        for p in SPECIES:
            if p != s and is_species_cluster(i, p):
                issues.append((s, i, f'label with foreign prefix {p}_ present in {s}'))
    if 'Unlabeled' in res_map:
        issues.append((s, 'Unlabeled', "'Unlabeled' was assigned to a group"))

report('E. node filter safe (no NN / Allen / foreign label grouped)', not issues,
       f'{len(issues)} issues')
for x in issues:
    print('   ', x)

## F. Edge robustness

`get_mapping_scores` fills one symmetric block per pair, so a reciprocal pair must read the same
score from both directions — a mismatch means the two CSVs came from different runs. The second
table lists best hits within `WINDOW` of `THRESH`: those edges (and the groups that hang off them)
would appear or vanish under a small threshold change, so they are where to look first if a group
looks wrong. Note the source notebook uses a strict `score > THRESH` for edges while its progress
print counts `>= THRESH`.

In [ ]:
WINDOW = 0.03

asym, borderline = [], []
for a, b in PAIRS:
    for src, dst in ((a, b), (b, a)):
        fwd, rev = mapping_frames[(src, dst)], mapping_frames[(dst, src)]
        for u, row in fwd.iterrows():
            v, score = row['SAMAP match'], float(row['SAMAP score'])
            recip = v in rev.index and rev.loc[v, 'SAMAP match'] == u
            if recip and not np.isclose(score, float(rev.loc[v, 'SAMAP score'])):
                asym.append({'u': u, 'v': v, 'fwd': score, 'rev': float(rev.loc[v, 'SAMAP score'])})
            if abs(score - THRESH) <= WINDOW:
                borderline.append({'from': u, 'to': v, 'score': round(score, 4),
                                   'reciprocal': recip,
                                   'edge': recip and score > THRESH,
                                   'group': res_map.get(u, '-')})

report('F. reciprocal scores symmetric across the two CSVs of a pair', not asym, f'{len(asym)} asymmetric')
if asym:
    display(pd.DataFrame(asym))

bl = (pd.DataFrame(borderline).drop_duplicates(subset=['from', 'to'])
      .sort_values('score').reset_index(drop=True))
print(f'\nbest hits within {WINDOW} of THRESH={THRESH} '
      f'({int(bl.edge.sum())} currently form an edge, {int((~bl.edge & bl.reciprocal).sum())} '
      f'are reciprocal but just below):')
display(bl)

# which groups would change if the threshold moved
for t in (THRESH - WINDOW, THRESH, THRESH + WINDOW):
    g = build_graph(mapping_frames, t)
    comps = [c for c in nx.connected_components(g) if len(c) > 1]
    print(f'  THRESH={t:.2f}: {g.number_of_edges()} edges, {len(comps)} groups, '
          f'{sum(len(c) for c in comps)} types grouped')

## G. Composition sanity

Not a pass/fail on correctness — these are the things worth eyeballing biologically. `dupes` means
two or more clusters *of the same species* were merged into one homology group, which happens
whenever a chain of reciprocal hits runs through another species; a group where one species
contributes almost all the cells is the other pattern to watch.

In [ ]:
comp = []
for name, mem in members.items():
    per = {s: sum(cell_counts[n] for n in mem if node_species[n] == s) for s in SPECIES}
    ntypes = {s: sum(1 for n in mem if node_species[n] == s) for s in SPECIES}
    tot = sum(per.values())
    comp.append({'group': assign_csv.get(mem[0], name),
                 'types': len(mem),
                 'species': sum(v > 0 for v in ntypes.values()),
                 'cells': tot,
                 **{f'{s}_cells': per[s] for s in SPECIES},
                 'dupes': ', '.join(f'{s}x{ntypes[s]}' for s in SPECIES if ntypes[s] > 1),
                 'max species share': round(max(per.values()) / tot, 2),
                 'min edge': round(min(d['weight'] for *_, d in G.subgraph(mem).edges(data=True)), 3),
                 'members': ' '.join(sorted(mem))})
comp = pd.DataFrame(comp).sort_values(['species', 'types'], ascending=False).reset_index(drop=True)
display(comp)

print('groups merging >1 cluster of the same species:',
      comp[comp.dupes != ''].group.tolist())
print('groups where one species holds >80% of the cells:',
      comp[comp['max species share'] > 0.8].group.tolist())
print('\ncells living in a cross-species group, per species:')
for s in SPECIES:
    a = ad.read_h5ad(H5AD[s], backed='r')
    o = a.obs[OUT_LEVEL].astype(str); del a
    ing = o.isin(set(assign_csv.values)).sum()
    print(f'  {s}: {ing}/{len(o)} ({100*ing/len(o):.1f}%)')

## H. Full mapping tables vs the written best-hit CSVs

The one check that reaches behind the pairwise CSVs to the SAMap output itself. Each
`Full_nmm_<pair>_<date>.csv` in `SAMAP_DIR` is an unrestricted `get_mapping_scores` block; applying
`eligible()` and `best_hits_from()` to it must reproduce that pair's two
`*_nonmam_mapping_SAMap_{DATE}.csv` exactly.

Orientation is auto-detected — the files are not written consistently (`cjdr` is rows=dr/cols=cj,
`cjac` and `acxt` are rows=cj/cols=ac) — and each axis is identified against the h5ad label columns,
so a table accidentally built on `OUT_LEVEL` instead of `LEVEL` fails loudly rather than looking
like a score drift. Pairs with no `Full_nmm_*` file yet are skipped.

In [ ]:
import glob

_labelsets = {}
def labelset(s, col):
    if (s, col) not in _labelsets:
        a = ad.read_h5ad(H5AD[s], backed='r')
        _labelsets[(s, col)] = set(a.obs[col].astype(str).unique()) if col in a.obs.columns else None
        del a
    return _labelsets[(s, col)]

CAND_COLS = ['ss_subclass_v5_nounlabeled_nn', 'ss_subclass_nounlabeled_nmm_v5_nn',
             'ss_subclass_nounlabeled_nmm_cl_v5_nn', 'ss_subclass_v5_nounlabeled']


def eligible(block):
    return block.loc[[i for i in block.index   if i[3:] in node_species],
                     [c for c in block.columns if c[3:] in node_species]]


def best_hits_from(block):
    out = {}
    for col in block.columns:
        item = col[3:]
        if block[col].isna().all() or block[col].max() == 0:
            out[item] = [None, 0.0]
        else:
            best = block[col].idxmax()
            out[item] = [best[3:], float(block.loc[best, col])]
    return out


def check_full_table(path):
    """-> (ok, list of row dicts). ok is None when the file cannot be interpreted."""
    name = os.path.basename(path)
    full = pd.read_csv(path, index_col=0)
    rp = {i[:2] for i in full.index}
    cp = {c[:2] for c in full.columns}
    if len(rp) != 1 or len(cp) != 1 or not (rp | cp) <= set(SPECIES):
        print(f'  {name}: uninterpretable prefixes rows={rp} cols={cp}')
        return None, []
    rs, cs = rp.pop(), cp.pop()

    rows = []
    ok = True
    for s, axis in ((rs, {i[3:] for i in full.index}), (cs, {c[3:] for c in full.columns})):
        hits = [c for c in CAND_COLS if labelset(s, c) == axis]
        good = LEVEL in hits
        ok &= good
        rows.append({'file': name, 'item': f'{s} axis', 'n': len(axis),
                     'ok': good, 'detail': ', '.join(hits) or 'NO KNOWN COLUMN'})

    v = full.values
    clean = int((v < 0).sum()) + int((v > 1).sum()) + int(np.isnan(v).sum())
    ok &= clean == 0
    zc = [c[3:] for c in full.columns if full[c].max() == 0]
    zr = [i[3:] for i in full.index   if full.loc[i].max() == 0]
    rows.append({'file': name, 'item': 'values in [0,1], no NaN', 'n': full.size,
                 'ok': clean == 0,
                 'detail': (f'all-zero cols {zc} rows {zr}' if (zc or zr) else '')})

    blk = eligible(full)
    for src, dst, block in ((cs, rs, blk), (rs, cs, blk.T)):
        fn = SAMAP_DIR + f'{src}_{dst}_nonmam_mapping_SAMap_{DATE}.csv'
        if not os.path.exists(fn):
            rows.append({'file': name, 'item': f'{src}->{dst}', 'n': 0,
                         'ok': None, 'detail': 'no reference CSV'})
            continue
        new = pd.DataFrame.from_dict(best_hits_from(block), orient='index',
                                     columns=['SAMAP match', 'SAMAP score']).sort_index()
        ref = pd.read_csv(fn, index_col=0).sort_index()
        idx_ok = list(new.index) == list(ref.index)
        j = new.join(ref, lsuffix='_new', rsuffix='_ref', how='inner')
        # a None best hit and the CSV round-trip NaN are the same thing
        mn = j['SAMAP match_new'].where(j['SAMAP match_new'].notna(), '').astype(str)
        mr = j['SAMAP match_ref'].where(j['SAMAP match_ref'].notna(), '').astype(str)
        md_ = int((mn != mr).sum())
        sd_ = int((~np.isclose(j['SAMAP score_new'], j['SAMAP score_ref'],
                               rtol=1e-9, atol=1e-12)).sum())
        good = idx_ok and md_ == 0 and sd_ == 0
        ok &= good
        dd = (j['SAMAP score_new'] - j['SAMAP score_ref']).abs()
        rows.append({'file': name, 'item': f'{src}->{dst}', 'n': len(j), 'ok': good,
                     'detail': (f'index {"same" if idx_ok else "DIFF"}, '
                                f'match diffs {md_}, score diffs {sd_}'
                                + (f', max|d| {dd.max():.2e}' if sd_ else ''))})
        if not good:
            m = mn != mr
            if m.any():
                display(j[m][['SAMAP match_new', 'SAMAP score_new',
                              'SAMAP match_ref', 'SAMAP score_ref']])
    return ok, rows


full_files = sorted(glob.glob(SAMAP_DIR + 'Full_nmm_*.csv'))
print(f'{len(full_files)} full mapping tables found\n')

all_rows, verdicts = [], {}
for f in full_files:
    v, rws = check_full_table(f)
    verdicts[os.path.basename(f)] = v
    all_rows += rws

if not full_files:
    report_ok = None
    RESULTS['H. full tables reproduce the best-hit CSVs'] = (None, 'no Full_nmm_*.csv present')
    print('SKIP  no Full_nmm_*.csv in SAMAP_DIR')
else:
    chk_h = pd.DataFrame(all_rows)
    display(chk_h)
    done = {k: v for k, v in verdicts.items() if v is not None}
    covered = sorted({tuple(sorted((r['item'].split('->')[0], r['item'].split('->')[-1])))
                      for r in all_rows if '->' in r['item']})
    report('H. full tables reproduce the best-hit CSVs',
           bool(done) and all(done.values()),
           f'{sum(bool(v) for v in done.values())}/{len(PAIRS)} pairs verified '
           f'({", ".join("-".join(p) for p in covered)})')
    missing = [f'{a}-{b}' for a, b in PAIRS
               if not any({a, b} == set(p) for p in covered)]
    if missing:
        print('  pairs with no full table yet:', ', '.join(missing))

### Appendix: blocks pickle (optional)

The same check against the cached `nonmam_samap_blocks_{DATE}.pkl` rather than the full CSVs. The
pickle was written under the SAMap env (pandas from py3.7); unpickling it under a modern pandas
raises `TypeError: Argument 'placement' has incorrect type`, so this only runs in the kernel that
produced it. It records nothing in the summary — check H above is the one that counts.

In [ ]:
try:
    with open(BLOCKS, 'rb') as fh:
        blocks = pickle.load(fh)
except Exception as e:
    blocks = None
    print(f'SKIP  blocks cache not loadable in this kernel: {type(e).__name__}: {e}')

if blocks is not None:
    rows = []
    for (a, b), blk in blocks.items():
        blk = eligible(blk)
        for (src, dst), block in (((a, b), blk), ((b, a), blk.T)):
            new = pd.DataFrame.from_dict(best_hits_from(block), orient='index',
                                         columns=['SAMAP match', 'SAMAP score']).sort_index()
            ref = mapping_frames[(src, dst)].sort_index()
            j = new.join(ref, lsuffix='_new', rsuffix='_ref', how='inner')
            mn = j['SAMAP match_new'].where(j['SAMAP match_new'].notna(), '').astype(str)
            mr = j['SAMAP match_ref'].where(j['SAMAP match_ref'].notna(), '').astype(str)
            rows.append({'pair': f'{src}->{dst}',
                         'same index': list(new.index) == list(ref.index),
                         'n': len(j),
                         'match diffs': int((mn != mr).sum()),
                         'score diffs': int((~np.isclose(j['SAMAP score_new'],
                                                         j['SAMAP score_ref'])).sum())})
    display(pd.DataFrame(rows))

## Summary

In [ ]:
summary = pd.DataFrame(
    [{'check': k,
      'result': 'PASS' if v[0] else ('SKIP' if v[0] is None else 'FAIL'),
      'note': v[1]}
     for k, v in RESULTS.items()])
display(summary)

failed = summary[summary.result == 'FAIL']
print('\n' + ('ALL CHECKS PASSED' if failed.empty
               else 'FAILURES: ' + ', '.join(failed.check)))